In [1]:
import re
import streamlit as st

from core.simulation import load_plecs
from core.llm import custom_responses
from core.optim import optimizers
from core.llm.llm import get_msg_history
from core.model_zoo.pann_dab import train_dab
import asyncio
from llama_index.core.agent.workflow import AgentWorkflow
from llama_index.core.tools import FunctionTool
from llama_index.llms.ollama import Ollama

d:\AI_CHO_NVH\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-04-08 14:59:15.902 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-08 14:59:15.904 WARNING streamlit.runtime.state.session_state_proxy: Session state does not function when running a script without `streamlit run`
2026-04-08 14:59:15.904 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-08 14:59:15.905 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-08 14:59:15.905 WARNIN

In [ ]:

from core.gui.design_stages import design_flow, other_tasks


asyncio.create_task(design_flow(agents, general_client, FlexRes=True)):
    """
        This is your customized design workflow
    """
    
    chat_engine0, chat_engine1, chat_engine2, agent_intent = agents
    
    # User textual input block for queries
    if prompt := st.chat_input("Your request:"):  # Prompt of user inputs and save to chat history
        st.session_state.messages.append({"role": "user", "content": prompt})
        with st.chat_message("user"):
            st.markdown(prompt)
        
        # save the historical messages into a list, to ensure that PE-GPT knows the chat history (LLM is memoryless in nature)
        messages_history = get_msg_history()
        # messages_history = [] # if no historical messages are used
        
        
        # The LLM agents are responsible for the following defined design tasks
        with st.chat_message("assistant"):
            response = await agent_intent.run(user_msg=f"""Please call the corresponding function based the user's Request given in square brackes '[]' 
                                         and Listen Carefully!! Only ONE function closest to the function descriptions should be called!!!!
                                         User's Request: [{prompt}]""".replace('\n', ''))
                                         # +"\n\nThe detailed function descriptions are defined below."
                                         # +"\n".join([description_task0, description_task1, description_task2,
                                         #             description_task3, description_task4, description_other_tasks]))
            
            
            
            
            if len(response.sources) >= 1: # if any function has been triggered
                if None in [item.raw_output for item in response.sources]: messages = other_tasks(general_client)
                else:
                    task = response.sources[0].raw_output # conduct the first matched task
                    
                    args = ()
                    if task == "Task 0":
                        kwargs = {"chat_engine": chat_engine1, "prompt": prompt, 
                                  "messages_history": messages_history}
                        response_pe = init_design(*args, **kwargs)
                        messages = [{"role": "assistant", "content": response_pe},]
                        
                    elif task == "Task 1":
                        kwargs = {"chat_engine": chat_engine1, "prompt": prompt, 
                                  "messages_history": messages_history}
                        messages = recommend_modulation(*args, **kwargs)
                        
                    elif task == "Task 2":
                        kwargs = {"chat_engine": chat_engine0, "prompt": prompt, 
                                  "messages_history": messages_history}
                        messages = evaluate_dab(*args, **kwargs) # these messages are predefined
                        if FlexRes:
                            prompt = """Attention!!! Now I have evaluated the current stress and soft switching
                            performances of the recommended modulation and the conventional SPS strategy, as the contents 
                            shown below. Please refer to your expertise for DAB modulations, completely rewrite the 
                            contents and provide more power electronics insights.""".replace("\n", "") + "\n" +\
                                "'"+"\n".join(item["content"] for item in messages)+"'"
                            response = chat_engine0.chat(prompt, chat_history=messages_history) 
                            st.write(response.response)
                            messages.append({"role": "assistant", "content": response.response})
                        
                    elif task == "Task 3":
                        kwargs = {}
                        messages = simulation_verification(*args, **kwargs)
                        
                    elif task == "Task 4":
                        kwargs = {"chat_engine": chat_engine2, "prompt": prompt}
                        messages = pe_gpt_introduction(*args, **kwargs)
                        
                    elif task == "Task 5":
                        kwargs = {}
                        messages = train_pann(*args, **kwargs)
                    
            else: # no function has been triggered
                messages = other_tasks(general_client)
                    
        for msg in messages:
            st.session_state.messages.append(msg) # Append the response to the message history


In [10]:
# from llama_index.llms.ollama import Ollama
# from llama_index.core.llms import ChatMessage, MessageRole
# rep = Ollama(model="mistral", base_url="http://localhost:11434", request_timeout=120.0).chat(messages=[ChatMessage(role=MessageRole.USER, content="Dân số Việt Nam hiện nay là bao nhiêu?")])
# print(rep)